<a href="https://colab.research.google.com/github/VukasinA/ML_projekti/blob/main/Zadatak4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zadatak 4.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from collections import Counter, defaultdict

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # ukloni URL-ove
    text = re.sub(r"@\w+", "", text)  # ukloni @username
    text = re.sub(r"#\w+", "", text)  # ukloni hashtagove
    text = re.sub(r"[^a-z\s]", "", text)  # ukloni sve sem slova
    text = re.sub(r"\s+", " ", text).strip()  # višestruki space -> jedan
    return text

# Učitaj skup podataka
df = pd.read_csv("disaster-tweets.csv")
df.dropna(subset=["text", "target"], inplace=True)

# Očisti tekstove
df["clean_text"] = df["text"].apply(clean_text)

# Podele podataka
X = df["clean_text"]
y = df["target"]

# Vectorizer (ograničen na 10k najčešćih reči)
vectorizer = CountVectorizer(max_features=10000)
X_vec = vectorizer.fit_transform(X)

accuracies = []
for _ in range(3):
    X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=np.random.randint(10000))
    model = MultinomialNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    accuracies.append(acc)

print(f"\nProsečna accuracy: {np.mean(accuracies):.4f}")

# ----------------------
# Analiza reči po klasama
# ----------------------

# Ponovo podela celog skupa
df['target'] = df['target'].astype(int)
positive_texts = df[df['target'] == 1]["clean_text"]
negative_texts = df[df['target'] == 0]["clean_text"]

# Tokenizacija svih reči
def tokenize(texts):
    words = []
    for text in texts:
        words.extend(text.split())
    return words

positive_words = tokenize(positive_texts)
negative_words = tokenize(negative_texts)

# Broj reči
pos_counter = Counter(positive_words)
neg_counter = Counter(negative_words)

# Top 5 reči
print("\nTop 5 reči u pozitivnim tvitovima (disaster=1):")
for word, count in pos_counter.most_common(5):
    print(f"{word}: {count}")

print("\nTop 5 reči u negativnim tvitovima (disaster=0):")
for word, count in neg_counter.most_common(5):
    print(f"{word}: {count}")

# ----------------------
# LR metrika (likelyhood ratio)
# ----------------------

# Samo reči koje se pojavljuju >=10 puta u obe klase
common_words = set([w for w in pos_counter if pos_counter[w] >= 10]) & set([w for w in neg_counter if neg_counter[w] >= 10])
lr_scores = {}

for word in common_words:
    lr = pos_counter[word] / neg_counter[word]
    lr_scores[word] = lr

sorted_lr = sorted(lr_scores.items(), key=lambda x: x[1], reverse=True)

print("\nTop 5 reči sa NAJVEĆIM LR (više u pozitivnim):")
for word, lr in sorted_lr[:5]:
    print(f"{word}: {lr:.2f}")

print("\nTop 5 reči sa NAJMANJIM LR (više u negativnim):")
for word, lr in sorted_lr[-5:]:
    print(f"{word}: {lr:.2f}")




Accuracy: 0.7905
Accuracy: 0.7814
Accuracy: 0.8024

Prosečna accuracy: 0.7914

Top 5 reči u pozitivnim tvitovima (disaster=1):
the: 1363
in: 1161
a: 933
of: 927
to: 757

Top 5 reči u negativnim tvitovima (disaster=0):
the: 1907
a: 1261
to: 1189
i: 1078
and: 918

Top 5 reči sa NAJVEĆIM LR (više u pozitivnim):
train: 5.64
pm: 5.56
fires: 5.31
fatal: 4.91
injured: 4.20

Top 5 reči sa NAJMANJIM LR (više u negativnim):
want: 0.19
let: 0.19
full: 0.12
love: 0.11
body: 0.11


Komentari:

Top reči u pozitivnim tvitovima često uključuju termine vezane za katastrofe, npr. 'fire', 'emergency', 'dead' dok u negativnim često imamo neutralne ili izraze o svakodnevici, npr. 'like', 'people', 'love'

LR metrika pokazuje relativni značaj reči za pozitivne vs negativne tvitove.

Reč sa velikim LR se mnogo češće pojavljuje u pozitivnim (katastrofe), dok sa malim LR više u negativnim.

Tako možemo identifikovati reči koje su "signal" za katastrofe i one koje su "signal" za obične tvitove.